In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import timesfm
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


c:\dev\ADIA-Lab-Structural-Break-Open-Benchmark-Challenge-Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded PyTorch TimesFM, likely because python version is 3.10.8 (tags/v3.10.8:aaaf517, Oct 11 2022, 16:50:30) [MSC v.1933 64 bit (AMD64)].


In [ ]:
# ==============================================================================
# 1. LOAD DATA
# ==============================================================================
print("Loading datasets...")
df_raw = pd.read_parquet('../data/raw/X_train.parquet').reset_index()

Loading datasets...


In [ ]:
# ==============================================================================
# 2A. TIMESFM INFERENCE & RAW PREDICTION PERSISTENCE
# ==============================================================================
print("Initializing Google TimesFM for full dataset inference...")

# 1. LOAD DATA & DYNAMIC HORIZON CALCULATION
df_raw = pd.read_parquet('../data/raw/X_train.parquet').reset_index()

# Calculate the max horizon based on the FULL dataset to configure the model
max_horizon_real = df_raw[df_raw['period'] == 1].groupby('id').size().max()
safe_horizon_len = max(128, int(np.ceil(max_horizon_real / 32.0)) * 32)

print(f"Max real horizon found in dataset: {max_horizon_real}")
print(f"Configuring TimesFM with dynamic horizon length: {safe_horizon_len}")

# 2. MODEL INITIALIZATION
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="cpu",
        per_core_batch_size=32,
        horizon_len=safe_horizon_len, 
        context_len=512, 
        num_layers=20,
        use_positional_embedding=True,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-1.0-200m-pytorch"
    )
)

# 3. FULL DATASET ITERATION
unique_ids = df_raw['id'].unique()

print(f"\nRunning PRODUCTION MODE with {len(unique_ids)} time series...")

list_of_dfs = []

for uid in tqdm(unique_ids, desc="TimesFM Inference (Full Dataset)"):
    series_data = df_raw[df_raw['id'] == uid].sort_values('time')
    
    pre_segment = series_data[series_data['period'] == 0]['value'].values
    
    post_df = series_data[series_data['period'] == 1].copy()
    post_segment = post_df['value'].values
    horizon_length = len(post_segment)
    
    # Skip series that lack enough context or have no valid post-break horizon
    if horizon_length < 2 or len(pre_segment) < 32:
        continue
        
    try:
        # Generate the forecast
        point_forecast, _ = tfm.forecast(inputs=[pre_segment], freq=[0])
        pred = point_forecast[0]
        
        # Saefty cut
        min_len = min(len(pred), len(post_segment))
        pred_trunc = pred[:min_len]
        
        # Truncate the dataframe to match and append the new TimesFM column
        post_df_trunc = post_df.iloc[:min_len].copy()
        post_df_trunc['value_timesfm'] = pred_trunc
        
        list_of_dfs.append(post_df_trunc)
                
    except Exception as e:
        # Fail gracefully without printing to avoid spamming the console
        pass

# 4. CONCATENATE AND SAVE (LONG FORMAT)
print("\nConcatenating results and saving to disk...")
df_long_preds = pd.concat(list_of_dfs, ignore_index=True)

# Rearrange columns to match standard tidy data format
df_long_preds = df_long_preds[['id', 'time', 'value', 'period', 'value_timesfm']]

raw_save_path = '../data/processed/timesfm_predictions.parquet'
df_long_preds.to_parquet(raw_save_path)

print(f"Production long-format predictions successfully saved to '{raw_save_path}'!")

Initializing Google TimesFM for full dataset inference...
Max real horizon found in dataset: 999
Configuring TimesFM with dynamic horizon length: 1024


Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 6636.56it/s]



Running PRODUCTION MODE with 10001 time series...


TimesFM Inference (Full Dataset): 100%|██████████| 10001/10001 [22:14:06<00:00,  8.00s/it]  



Concatenating results and saving to disk...
Production long-format predictions successfully saved to '../data/processed/timesfm_predictions_long_production.parquet'!
